# 09 — Feature diagram για poster

Παράγει ένα **plotly figure** (similar to Jönsson & Eklundh 2004 Fig. 8) δείχνοντας όλα τα phenometric features σε **μία πραγματική καμπύλη GCVI (Green Chlorophyll Vegetation Index)** από το δικό μας dataset.

Annotations:
- **SoS** (Start of Season)
- **LeftShoulder (L)** — flag leaf proxy
- **POS** (Peak of Season)
- **Amplitude** (peak − base)
- **Integrated** (AUC, shaded area)
- **Green-up rate** (slope arrow)
- **Duration of green-up**

Output: SVG για poster.

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
import plotly.graph_objects as go

RAW = 'data/processed/buffer/hls_phenology_merged.parquet'
FEAT = 'data/processed/features/features_phenometrics.parquet'

raw = pd.read_parquet(RAW)
feat = pd.read_parquet(FEAT)

GOLD = '#CEB888'
BLACK = '#000000'
DARK = '#6B5C36'
LIGHT = '#F0E8CC'

print(f'Raw: {len(raw):,} rows')
print(f'Features: {len(feat)} field-years')

## 1. Pick a clean example field-year

Select a field-year με:
- Dense satellite observations (n_obs > 25)
- All features successfully extracted (no NaN)
- Clean GCVI profile (low noise)

In [ ]:
# Candidates: complete feature records + all criteria
feat_complete = feat.dropna(subset=['GCVI_SOS','GCVI_POS','GCVI_LeftShoulder','GCVI_peak','GCVI_base',
                                     'GCVI_amplitude','GCVI_integrated','GCVI_greenup_rate',
                                     'GCVI_duration_greenup','GCVI_greenup_midpoint',
                                     'DL_c3_greenup_steepness','flag_true_doy'])
feat_complete = feat_complete[feat_complete['n_obs'] > 30].copy()
print(f'Candidates (n_obs>30, DL success, all features valid): {len(feat_complete)}')

# Biological correctness: flag_true_doy should be between SOS and POS
# Flag leaf must be clearly BEFORE the peak (on the green-up slope, visible on left)
feat_complete = feat_complete[
    (feat_complete['flag_true_doy'] >= feat_complete['GCVI_SOS'] + 5) &
    (feat_complete['flag_true_doy'] <= feat_complete['GCVI_POS'] - 10)
]
print(f'After bio filter (SOS <= flag_obs <= POS): {len(feat_complete)}')

# Amplitude > 1.5 (GCVI amplitude) to avoid sparse curves
feat_complete = feat_complete[feat_complete['GCVI_amplitude'] > 1.5]

# LeftShoulder should be close to flag_true_doy
feat_complete['L_flag_diff'] = np.abs(feat_complete['GCVI_LeftShoulder'] - feat_complete['flag_true_doy'])

# Typical green-up duration (30-90 days)
feat_complete = feat_complete[
    (feat_complete['GCVI_duration_greenup'] > 30) & 
    (feat_complete['GCVI_duration_greenup'] < 90)
]

# Reasonable SOS timing (March-April)
feat_complete = feat_complete[(feat_complete['GCVI_SOS'] > 60) & (feat_complete['GCVI_SOS'] < 110)]
feat_complete = feat_complete[(feat_complete['GCVI_POS'] > 130) & (feat_complete['GCVI_POS'] < 170)]

print(f'After all filters: {len(feat_complete)}')

# Rank: prefer low L-flag diff + reasonable amplitude + median-like SOS
feat_complete['score'] = (feat_complete['L_flag_diff'] * 3.0 + 
                           np.abs(feat_complete['GCVI_SOS'] - 85) * 0.5 +
                           np.abs(feat_complete['flag_true_doy'] - 112) * 0.5)
feat_complete = feat_complete.sort_values('score')

print('\nTop 10 candidates:')
print(feat_complete[['field_id','year','flag_true_doy','GCVI_SOS','GCVI_LeftShoulder','GCVI_POS',
                     'GCVI_amplitude','n_obs','L_flag_diff']].head(10).round(2).to_string(index=False))

# Pick best — manual override for best visual example
OVERRIDE_FID, OVERRIDE_YR = 'BUF_02363', 2016
match = feat_complete[(feat_complete['field_id']==OVERRIDE_FID) & (feat_complete['year']==OVERRIDE_YR)]
if len(match) > 0:
    selected = match.iloc[0]
    print(f'\nUsing override: {OVERRIDE_FID} ({OVERRIDE_YR})')
else:
    selected = feat_complete.iloc[0]
    print(f'\nOverride not in candidates, using best ranked: {selected["field_id"]}')
FID, YR = selected['field_id'], int(selected['year'])
print(f'\nSelected: {FID}, year {YR}')
print(f'  flag_true_doy = {selected["flag_true_doy"]:.0f}')
print(f'  GCVI_SOS      = {selected["GCVI_SOS"]:.0f}')
print(f'  GCVI_L        = {selected["GCVI_LeftShoulder"]:.0f}')
print(f'  GCVI_POS      = {selected["GCVI_POS"]:.0f}')
print(f'  L_flag_diff   = {selected["L_flag_diff"]:.1f} days (target: small)')

In [ ]:
# Load NDVI time series for selected field-year
sub = raw[(raw['field_id']==FID) & (raw['year']==YR) & (raw['GCVI'].notna())]
obs_doys = sub['doy'].values
obs_ndvi = sub['GCVI'].values

# Smooth (same as notebook 04)
idx = np.argsort(obs_doys)
doys_s = np.array(obs_doys)[idx]
vals_s = np.array(obs_ndvi)[idx]
u_doys = np.unique(doys_s)
u_vals = np.array([vals_s[doys_s == d].mean() for d in u_doys])
target = np.arange(1, 366)
daily = np.interp(target, u_doys, u_vals)
win = 15
smoothed = savgol_filter(daily, win, 2)

print(f'Obs: {len(obs_doys)} satellite observations')
print(f'DOY range: {obs_doys.min()}–{obs_doys.max()}')

## 2. Extract all features for this specific curve

In [ ]:
# All features from the pre-computed parquet
f = {
    'base':          selected['GCVI_base'],
    'peak':          selected['GCVI_peak'],
    'amplitude':     selected['GCVI_amplitude'],
    'SOS':           int(selected['GCVI_SOS']),
    'POS':           int(selected['GCVI_POS']),
    'midpoint':      selected['GCVI_greenup_midpoint'],
    'LeftShoulder':  int(selected['GCVI_LeftShoulder']),
    'greenup_rate':  selected['GCVI_greenup_rate'],
    'duration':      selected['GCVI_duration_greenup'],
    'integrated':    selected['GCVI_integrated'],
    'flag_obs':      int(selected['flag_true_doy']),
}
for k, v in f.items(): print(f'  {k:15s}: {v}')

## 3. Build the figure

In [ ]:
fig = go.Figure()

# (1) Integrated area — shaded polygon under curve from SOS to POS, above baseline
sos_i, pos_i = f['SOS']-1, f['POS']-1
area_x = list(target[sos_i:pos_i+1]) + list(target[sos_i:pos_i+1])[::-1]
area_y = list(smoothed[sos_i:pos_i+1]) + [f['base']] * (pos_i - sos_i + 1)
fig.add_trace(go.Scatter(
    x=area_x, y=area_y, fill='toself', fillcolor='rgba(206, 184, 136, 0.25)',
    line=dict(color='rgba(0,0,0,0)'), showlegend=True,
    name=f'Integrated (AUC) ≈ {f["integrated"]:.1f}', hoverinfo='skip'))

# (2) Raw satellite observations
fig.add_trace(go.Scatter(
    x=obs_doys, y=obs_ndvi, mode='markers',
    marker=dict(size=7, color='rgba(0,0,0,0.55)', symbol='circle',
                line=dict(color=BLACK, width=0.5)),
    name='Raw GCVI (HLS L8+S2)'))

# (3) Smoothed curve
fig.add_trace(go.Scatter(
    x=target, y=smoothed, mode='lines',
    line=dict(color=BLACK, width=2.5), name='Smoothed GCVI (Savitzky-Golay)'))

# (4) Baseline (horizontal dashed)
fig.add_hline(y=f['base'], line=dict(color=DARK, width=1, dash='dot'), opacity=0.6,
              annotation_text=f'base = {f["base"]:.2f}', annotation_position='left')

# (5) Peak level (horizontal dashed)
fig.add_hline(y=f['peak'], line=dict(color=DARK, width=1, dash='dot'), opacity=0.6,
              annotation_text=f'peak = {f["peak"]:.2f}', annotation_position='left')

# (6) Amplitude arrow (double-headed, right side)
arrow_x = 200
fig.add_annotation(x=arrow_x, y=f['peak'], ax=arrow_x, ay=f['base'],
                   xref='x', yref='y', axref='x', ayref='y',
                   showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2,
                   arrowcolor=BLACK)
fig.add_annotation(x=arrow_x, y=f['base'], ax=arrow_x, ay=f['peak'],
                   xref='x', yref='y', axref='x', ayref='y',
                   showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2,
                   arrowcolor=BLACK)
fig.add_annotation(x=arrow_x+3, y=(f['peak']+f['base'])/2,
                   text=f'<b>Amplitude<br>{f["amplitude"]:.2f}</b>',
                   showarrow=False, xanchor='left', font=dict(size=12, color=BLACK))

# (7) Vertical marker lines + labels for SOS, LeftShoulder, POS, flag observed
markers = [
    ('SoS',         f['SOS'],          smoothed[sos_i],      '#888',   'SoS (DOY {})'),
    ('L (flag leaf)', f['LeftShoulder'], smoothed[f['LeftShoulder']-1], GOLD, 'LeftShoulder (DOY {})'),
    ('POS',         f['POS'],          f['peak'],            '#666',   'POS (DOY {})'),
]
for label, doy, yv, col, tmpl in markers:
    fig.add_trace(go.Scatter(x=[doy, doy], y=[f['base'], yv],
                             mode='lines', line=dict(color=col, width=2, dash='dash'),
                             showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=[doy], y=[yv], mode='markers+text',
                             marker=dict(size=14, color=col, line=dict(color=BLACK, width=1.5)),
                             text=[f' {label}'], textposition='top right',
                             textfont=dict(size=14, color=BLACK),
                             name=tmpl.format(doy)))

# (8) Observed flag leaf — vertical line with star
fig.add_trace(go.Scatter(x=[f['flag_obs'], f['flag_obs']], y=[0, f['peak']*1.1],
                         mode='lines', line=dict(color='#D41B2C', width=2, dash='dashdot'),
                         showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=[f['flag_obs']], y=[f['peak']*1.05], mode='markers+text',
                         marker=dict(size=16, color='#D41B2C', symbol='star', line=dict(color=BLACK, width=1)),
                         text=[f' Observed flag leaf (DOY {f["flag_obs"]})'], textposition='middle right',
                         textfont=dict(size=13, color='#D41B2C'),
                         name=f'Observed flag leaf (DOY {f["flag_obs"]})'))

# (9) Green-up rate — annotation pointing to steepest slope
slope_doy = (f['SOS'] + f['POS']) // 2
slope_x = slope_doy
slope_y = smoothed[slope_doy-1]
fig.add_annotation(x=slope_x, y=slope_y,
                   text=f'Green-up rate<br>{f["greenup_rate"]:.4f}/day',
                   xref='x', yref='y',
                   ax=slope_x+25, ay=slope_y-0.18*f['amplitude'], axref='x', ayref='y',
                   showarrow=True, arrowhead=2, arrowcolor=DARK,
                   font=dict(size=11, color=DARK), bgcolor=LIGHT,
                   bordercolor=DARK, borderwidth=1, borderpad=3)

# (10) Duration of green-up (SOS → POS) horizontal bracket at bottom
bracket_y = f['base'] - 0.07*f['amplitude']
fig.add_annotation(x=f['POS'], y=bracket_y, ax=f['SOS'], ay=bracket_y,
                   xref='x', yref='y', axref='x', ayref='y',
                   showarrow=True, arrowhead=3, arrowsize=1, arrowwidth=2, arrowcolor=BLACK)
fig.add_annotation(x=f['SOS'], y=bracket_y, ax=f['POS'], ay=bracket_y,
                   xref='x', yref='y', axref='x', ayref='y',
                   showarrow=True, arrowhead=3, arrowsize=1, arrowwidth=2, arrowcolor=BLACK)
fig.add_annotation(x=(f['SOS']+f['POS'])/2, y=bracket_y - 0.04*f['amplitude'],
                   text=f'<b>Duration of green-up = {int(f["duration"])} days</b>',
                   showarrow=False, font=dict(size=12, color=BLACK))

# Layout
fig.update_layout(
    title=dict(text='GCVI phenology features',
               font=dict(size=20, color=BLACK, family='Inter, Arial, sans-serif')),
    xaxis=dict(title=dict(text='Day of Year (DOY) — Winter Wheat Growing Season', font=dict(size=15)),
               range=[30, 210], gridcolor='#eee', color=BLACK,
               tickfont=dict(size=13),
               tickvals=[30, 60, 90, 120, 150, 180, 210],
               ticktext=['Feb 1', 'Mar 1', 'Apr 1', 'May 1', 'Jun 1', 'Jul 1', 'Jul 30']),
    yaxis=dict(title=dict(text='GCVI', font=dict(size=15)),
               range=[f['base']-0.15*f['amplitude'], f['peak']+0.15*f['amplitude']],
               gridcolor='#eee', color=BLACK, tickfont=dict(size=13)),
    plot_bgcolor='white',
    width=1100, height=600,
    font=dict(family='Inter, Arial, sans-serif', size=13),
    margin=dict(l=70, r=60, t=70, b=70),
    legend=dict(yanchor='top', y=0.98, xanchor='left', x=0.02,
                bgcolor='rgba(255,255,255,0.85)', bordercolor=DARK, borderwidth=1,
                font=dict(size=11))
)

fig.show()

# Save SVG for poster
out = 'poster_figures/feature_diagram_gcvi.svg'
fig.write_image(out, format='svg')
print(f'\nSaved: {out}')
fig.write_html(out.replace('.svg', '.html'))
print(f'Also: {out.replace(".svg", ".html")}')